# Classification de monuments
### Taj Mahal / Grande Muraille de Chine / Christ Rédempteur

**Projet Annuel — ESGI 3ème année — Machine Learning (N. Vidal)**

Ce notebook documente notre démarche, nos implémentations et l'analyse de nos résultats, modèle par modèle. Conformément au syllabus, chaque algorithme est d'abord validé sur des cas de test classiques, puis appliqué à notre problématique (classification d'images de 3 monuments)

## 1. Modèle linéaire (Perceptron — règle de Rosenblatt)

### 1.1 Rappel théorique

Notre problématique (classer une image en 3 catégories) est un problème
de classification. 
Nous avons donc choisi de n'implémenter que la partie classification du modèle linéaire, et pas la régression.

Le modèle linéaire de classification, c'est le Perceptron : il trace
une frontière (droite en 2D, hyperplan en dimension supérieure) qui
sépare les exemples en deux classes. La sortie est en -1 / +1.

**Comment apprendre les poids $W$ ?** Nous utilisons la règle de
Rosenblatt (cours, slides "Apprendre — Modèle Linéaire et PMC", p.65) :

$$W \leftarrow W + \alpha \,(Y_k - g(X_k))\, X_k$$

On tire un exemple $k$ au hasard, on regarde l'écart entre la sortie
attendue $Y_k$ et la sortie obtenue $g(X_k)$, et on déplace $W$ un peu
dans la bonne direction. 
On répète cette opération un grand nombre de fois.

### 1.2 Multi-classes : one-vs-rest

Notre problème a **3 classes**, alors qu'un perceptron simple ne sait
séparer que 2 classes (-1 / +1). Nous utilisons donc la stratégie
**one-vs-rest** : un perceptron par classe (donc 3 perceptrons), chacun
entraîné à répondre *"+1 si c'est ma classe, -1 sinon"*. À la
prédiction, on garde la classe dont le perceptron est le plus confiant
(le score $W \cdot X$ le plus élevé).

C'est la même approche que celle illustrée dans le notebook de cas de
tests fourni par M. Vidal (*"Multi Linear 3 classes : Linear Model x3"*).

### 1.3 Implémentation

Le modèle est entièrement implémenté **en C** (`lib/src/linear_model.c`),
conformément au syllabus, et appelé depuis Python via `ctypes`
(`python_api/ml_bridge.py`). Aucune bibliothèque externe (scikit-learn,
etc.) n'est utilisée pour l'algorithme lui-même.

In [6]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import Markdown, display

from python_api.ml_bridge import LinearModel
from preprocessing.image_processor import ImageProcessor

np.random.seed(42)

In [7]:
# Validation rapide sur cas classiques (cf. notebook de cas de tests du cours)
X_simple = [[1,1],[2,2],[2,3],[3,2],[5,5],[6,6],[7,5],[6,7]]
y_simple = [0,0,0,0,1,1,1,1]
m1 = LinearModel(n_features=2, n_classes=2)
m1.train(X_simple, y_simple, learning_rate=0.01, n_iterations=20000)
acc1 = np.mean([m1.predict(x) for x in X_simple] == np.array(y_simple))

X_xor = [[0,0],[0,1],[1,0],[1,1]]
y_xor = [0,1,1,0]
m2 = LinearModel(n_features=2, n_classes=2)
m2.train(X_xor, y_xor, learning_rate=0.01, n_iterations=20000)
acc2 = np.mean([m2.predict(x) for x in X_xor] == np.array(y_xor))

display(Markdown(f"""
**Cas linéairement séparable : {acc1:.0%}** — **Cas XOR : {acc2:.0%}**
(échec attendu pour XOR, non linéairement séparable : limite connue du modèle, qui motive le passage au PMC)
"""))

Bibliothèque chargée: c:\Users\MIMI\OneDrive - Reseau-GES\Documents\ESGI_3eme_Annee\Projet Annuel\ProjetClassification\lib\libml.dll


AttributeError: function 'linear_predict_scores' not found

### Application au dataset (3 monuments)

In [ ]:
import tqdm as _t
import preprocessing.image_processor as _ip
_ip.tqdm = lambda *a, **k: _t.tqdm(*a, **{**k, 'disable': True})

processor = ImageProcessor()
X, y = processor.load_dataset('../data/raw', normalize=True)

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

print(f"Dataset : {len(X)} images ({dict(zip(processor.class_names, np.bincount(y)))})")
print(f"Train : {len(X_train)} | Test : {len(X_test)}")

In [ ]:
model = LinearModel(n_features=X.shape[1], n_classes=3)
model.train(X_train.tolist(), y_train.tolist(), learning_rate=0.01, n_iterations=50000)

y_pred = np.array([model.predict(x) for x in X_test.tolist()])
accuracy = np.mean(y_pred == y_test)
print(f"\nPrécision sur le test : {accuracy:.2%}")
print(classification_report(y_test, y_pred, target_names=processor.class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 3))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), processor.class_names, rotation=20)
plt.yticks(range(3), processor.class_names)
plt.xlabel("Prédiction"); plt.ylabel("Vérité")
plt.title("Matrice de confusion — Linéaire")
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

**Conclusion :** le modèle linéaire plafonne autour de XX% sur nos données réelles (vs 33% au hasard sur 3 classes). C'est cohérent avec sa limite théorique (cf. XOR ci-dessus) : une image n'est pas linéairement séparable en pixels bruts. Ce résultat sert de base pour comparer avec le PMC, attendu plus performant.